# Holdie — אימון מסווג אודיו ב-Google Colab

המחברת הזו מבצעת מקצה-לקצה: קלון של הריפו, חיתוך האודיו לחתיכות של 5 שניות, אימון על GPU, ואז שמירת ה-checkpoints חזרה ל-Drive.

**לפני שמתחילים:**
1. `Runtime → Change runtime type → GPU` (T4 חינמי מספיק להתחלה; A100/L4 ירוצו מהר יותר).
2. סדרו את האודיו הגולמי ב-Google Drive במבנה הבא:

```
MyDrive/holdie-audio/raw/
├── human/
│   ├── he/      ← הקלטות נציג בעברית
│   ├── en/      ← הקלטות נציג באנגלית (אופציונלי)
│   └── ar/      ← וכו'
├── ivr/
│   ├── he/
│   └── en/
├── music/
│   └── n_a/     ← למוזיקה השפה לא רלוונטית
└── recording/
    ├── he/
    └── en/
```

שמות תתי-תיקיות השפות הם קודי ISO 639-1 (`he`, `en`, `ar`, `ru` ...). למוזיקה השתמשו ב-`n_a` (קולון או סלאש אסורים בשמות תיקיות; הסקריפט יתרגם אותם ל-`n/a` ב-CSV).

אפשר להוסיף שפות חדשות אחר כך — פשוט להוסיף תתי-תיקיות ולהריץ שוב את תאי החיתוך.

## 1. בדיקת ה-GPU

In [ ]:
!nvidia-smi

## 2. קלון של הריפו והתקנת תלויות

נשתמש בענף `claude/multilingual-training-support-FXitV` שבו תמיכת ריבוי-השפות. החליפו ל-`main` לאחר מיזוג.

In [ ]:
REPO_URL = "https://github.com/YossiYad/research.git"
BRANCH = "claude/multilingual-training-support-FXitV"

import os, subprocess
if not os.path.exists("/content/research"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "/content/research"], check=True)
else:
    subprocess.run(["git", "-C", "/content/research", "pull", "--ff-only"], check=True)

%cd /content/research

In [ ]:
!pip install -q -r requirements.txt

## 3. חיבור Google Drive

ה-Drive נטען ל-`/content/drive`. אם זו הפעם הראשונה, ייפתח חלון אישור.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. הגדרת נתיבים

אם בחרתם מבנה תיקיות אחר ב-Drive — עדכנו את `DRIVE_RAW` כאן.

In [ ]:
from pathlib import Path

DRIVE_RAW = Path("/content/drive/MyDrive/holdie-audio/raw")
DRIVE_OUT = Path("/content/drive/MyDrive/holdie-audio/checkpoints")

PROJECT = Path("/content/research")
LOCAL_RAW = PROJECT / "dataset" / "raw"
LOCAL_PROCESSED = PROJECT / "dataset" / "processed"
METADATA = PROJECT / "dataset" / "metadata.csv"
CHECKPOINTS = PROJECT / "checkpoints"

assert DRIVE_RAW.exists(), f"לא נמצא {DRIVE_RAW} — ודאו שהעליתם את האודיו ל-Drive במבנה הנכון"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("DRIVE_RAW:", DRIVE_RAW)
print("DRIVE_OUT:", DRIVE_OUT)

## 5. חיתוך האודיו (אוטומטי לכל קטגוריה ושפה)

התא הזה סורק את `DRIVE_RAW/{class}/{lang}/`, חותך כל תיקייה לחתיכות של 5 שניות, ושומר ב-`dataset/processed/{class}/`. עמודת ה-`language` נכתבת ל-`metadata.csv` אוטומטית.

שם תיקיית שפה `n_a` (אסור סלאש בשם תיקייה) מתורגם ל-`n/a` ב-CSV. כל קוד אחר נשמר כפי שהוא.

In [ ]:
import subprocess, sys

VALID_CLASSES = ["human", "ivr", "music", "recording"]

# מחיקה של metadata.csv ישן כדי שהריצה תהיה דטרמיניסטית
if METADATA.exists():
    print(f"מוחק {METADATA} ישן ובונה מחדש")
    METADATA.unlink()

for cls in VALID_CLASSES:
    cls_dir = DRIVE_RAW / cls
    if not cls_dir.exists():
        print(f"⏭ {cls}: אין תיקייה ב-Drive, מדלג")
        continue

    lang_dirs = [d for d in sorted(cls_dir.iterdir()) if d.is_dir()]
    if not lang_dirs:
        # תאימות לאחור — אם אין תתי-תיקיות שפה, חותכים בלי שפה (unknown)
        print(f"⚠ {cls}: אין תתי-תיקיות שפה, חותך עם language=unknown")
        subprocess.run([
            sys.executable, "scripts/chop_audio.py",
            "--input", str(cls_dir),
            "--class", cls,
            "--language", "unknown",
        ], check=True)
        continue

    for lang_dir in lang_dirs:
        lang_code = lang_dir.name.replace("n_a", "n/a")
        print(f"\n=== {cls} / {lang_code} ===")
        subprocess.run([
            sys.executable, "scripts/chop_audio.py",
            "--input", str(lang_dir),
            "--class", cls,
            "--language", lang_code,
        ], check=True)

In [ ]:
# סיכום מהיר של מה שהתקבל
import sys
sys.path.insert(0, str(PROJECT / "scripts"))
from dataset import load_metadata, class_distribution, language_distribution, class_language_distribution

meta = load_metadata(METADATA)
print(f"סה\"כ חתיכות: {len(meta)}")
print("לפי קטגוריה:", class_distribution(meta))
print("לפי שפה:    ", language_distribution(meta))
print("\nלפי (קטגוריה, שפה):")
for (cls, lang), n in sorted(class_language_distribution(meta).items()):
    print(f"  {cls:>10} / {lang:>6}: {n}")

## 6. אימון

בחירת מודל הבסיס:
- **`facebook/wav2vec2-base`** — 95M פרמטרים, מהיר, אומן על אנגלית. מתאים להתחלה ולסט הנתונים קטן.
- **`facebook/wav2vec2-xls-r-300m`** — 300M פרמטרים, רב-לשוני (53 שפות כולל עברית). מומלץ למאגר בכמה שפות.
- **`imvladikon/wav2vec2-xls-r-300m-hebrew`** — מותאם עברית, טוב במיוחד אם רוב המאגר עברית.

ההמלצה בריבוי שפות: `xls-r-300m`. בעברית-בלבד: ה-checkpoint העברי.

התהליך מדפיס התפלגות פר-שפה ב-train/val/test, ובסיום נקבל גם **דיוק בדיקה פר שפה** — כך תוכלו לראות אם המודל מציג ביצועים אחידים בין שפות או מוטה לאחת מהן.

In [ ]:
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-4

!python scripts/train.py \
  --base-model {BASE_MODEL} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR}

## 7. ויזואליזציה

מציג גרף הפסד/דיוק לאורך האפוקים ומטריצת בלבול.

In [ ]:
!python scripts/visualize.py

## 8. שמירה חזרה ל-Drive

ה-checkpoints נשמרים גם מקומית (תיאבדו עם סשן Colab) — חשוב להעתיק אותם ל-Drive.

In [ ]:
import shutil
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
dest = DRIVE_OUT / f"run-{stamp}"
dest.mkdir(parents=True, exist_ok=True)

for name in ("best.pt", "last.pt", "history.json"):
    src = CHECKPOINTS / name
    if src.exists():
        shutil.copy2(src, dest / name)
        print(f"✓ {name} → {dest / name}")

# גם metadata.csv לתיעוד מלא של המאגר שהמודל אומן עליו
if METADATA.exists():
    shutil.copy2(METADATA, dest / "metadata.csv")
    print(f"✓ metadata.csv → {dest / 'metadata.csv'}")

print(f"\nכל הריצה נשמרה ב: {dest}")